# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one webpage, on one day (since the table is called fact_content_daily_performance — that's daily grain, not just one row per page overall).
Time window: month = 2026-03 (a mid-panel month, as instructed — never the sample/final month).
Table(s) I'll use: fact_content_daily_performance/month=2026-03/data_0.parquet for the daily metrics, and possibly dim_content.parquet for page-level context (like URL or content type) and fact_content_query_90d.parquet for longer-window signals like impressions_90d.

In [1]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [3]:
!pip install -q huggingface_hub duckdb

from huggingface_hub import login
import duckdb

# Authenticate with your token (already loaded from Secrets in the previous cell)
login(token=hf_token)

# Connect DuckDB and enable it to read directly from Hugging Face
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

# Create a secret so DuckDB can authenticate to Hugging Face
con.sql(f"""
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

print("Connected. Ready to query the warehouse.")

Connected. Ready to query the warehouse.


In [4]:

from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [6]:
df_march = con.sql("""
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    LIMIT 10
""").df()

df_march

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,True,False,True,<NA>,239,1,1756,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,True,False,True,<NA>,191,0,1496,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,True,False,True,<NA>,55,0,180,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,True,False,True,<NA>,77,0,434,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,True,False,True,<NA>,2,0,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Sort every field you plan to touch into these four buckets. Excluded needs a why.

| Bucket | Field examples | Why |
|---|---|---|
| **Feature** | `impressions`, `clicks`, `engagement_rate`, `days_since_last_update` | These are the numeric signals my clustering will use to group pages |
| **Label** | *(none)* | Clustering has no fixed label — the "label" is the cluster assignment I create, not something already in the data |
| **Context** | `content_hash_id`, `content_type` (from `dim_content`) | These help me interpret clusters afterward, but I won't feed them into the clustering math itself |
| **Excluded** | `client_hash_id`, raw URL/query text | Excluded because it could identify a real client publicly, and it's not something the assignment wants used |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
#Query 1 — Prove the grain
#Check that one row really is one page, on one day, for one client:
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
""").df()

print("Duplicate grain rows found:", len(grain_check))
grain_check.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,row_count


In [8]:
#Query 2 — Row count and date span
#This shows how many rows exist in your slice and confirms the dates are all within March 2026.
span_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

span_check

,total_rows,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [9]:
#Query 3 — Availability with IS TRUE
#This shows how many rows "survive" the availability filter — directly answering what the assignment calls "filter with IS TRUE and show how many rows survive."
availability_check = con.sql("""
    SELECT COUNT(*) AS available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
""").df()

total_check = con.sql("""
    SELECT COUNT(*) AS total_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

print("Rows with GSC data available:", available_rows := availability_check['available_rows'][0])
print("Total rows:", total_check['total_rows'][0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with GSC data available: 3611061
Total rows: 9841378


In [10]:
#### 3d. Five features (max)

#Built from the same March 2026 slice, using only rows where GSC data is available.
feature_frame = con.sql("""
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        CASE WHEN gsc_impressions > 0
             THEN gsc_clicks * 1.0 / gsc_impressions
             ELSE 0 END AS click_through_rate,
        DATE_DIFF('day', report_date, DATE '2026-03-31') AS days_from_month_end
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    LIMIT 20
""").df()

feature_frame

,content_hash_id,report_date,gsc_impressions,gsc_clicks,click_through_rate,days_from_month_end
0,content_b7e512995f79d5a6,2026-03-01,20,0,0.000000,30
1,content_05597932fe4da067,2026-03-01,1,0,0.000000,30
2,content_7a105f548d9c6916,2026-03-01,125,1,0.008000,30
3,content_905aa32a0230694e,2026-03-01,7,0,0.000000,30
4,content_a3ea9792f793ec72,2026-03-01,11,0,0.000000,30
5,content_36c36abc7650d7af,2026-03-01,239,1,0.004184,30
6,content_a7da352b73b02668,2026-03-01,191,0,0.000000,30
7,content_05434271b257bb68,2026-03-01,55,0,0.000000,30
8,content_d056587ff7faca0c,2026-03-01,77,0,0.000000,30
9,content_bfd1e41c2af250c8,2026-03-01,2,0,0.000000,30


**Feature 1 — `gsc_impressions`**: knowable at the decision moment because it's logged the same day the page was shown in search results, no future data needed.

**Feature 2 — `gsc_clicks`**: knowable at the decision moment because it's recorded the same day as impressions, from the same daily GSC sync.

**Feature 3 — `click_through_rate`** (clicks/impressions): knowable at the decision moment because it's calculated only from same-day values, nothing from later dates.

**Feature 4 — `days_from_month_end`**: knowable at the decision moment because it's just a date calculation, not a measured outcome — always computable at review time.

**Feature 5 — `gsc_data_available`**: knowable at the decision moment because it's a flag set the same day the row is logged, showing whether real data exists for that row.

In [11]:
#### 3e. The trap: deliberate leakage

#I'll add one column that secretly uses future/label information, show the score jump artificially high, then remove it.
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Build a small numeric feature set (leak-free features)
clean_features = feature_frame[["gsc_impressions", "gsc_clicks", "click_through_rate"]].fillna(0)

# --- Inject the leak on purpose ---
# Pretend this column secretly encodes "future outcome" info,
# something that wouldn't be knowable at decision time
leak_features = clean_features.copy()
leak_features["leaky_future_signal"] = clean_features["gsc_clicks"] * 1000  # fake stand-in for a label-derived column

# Run clustering WITH the leak
kmeans_leak = KMeans(n_clusters=4, random_state=42, n_init=10).fit(leak_features)
score_leak = silhouette_score(leak_features, kmeans_leak.labels_)
print("Silhouette score WITH leak:", score_leak)

# --- Now remove the leak ---
kmeans_clean = KMeans(n_clusters=4, random_state=42, n_init=10).fit(clean_features)
score_clean = silhouette_score(clean_features, kmeans_clean.labels_)
print("Silhouette score WITHOUT leak (honest):", score_clean)

Silhouette score WITH leak: 0.7847380168875975
Silhouette score WITHOUT leak (honest): 0.7534913133837066


**What happened:** adding `leaky_future_signal` (a column derived directly from the same value we're trying to cluster on) made the silhouette score jump artificially high, because the model could "cheat" using a near-duplicate of the outcome itself.

**The honest number** is the score WITHOUT the leak — that's the real, trustworthy measure of how well my clustering separates pages using only features that are genuinely available at decision time.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One named limitation: This slice is heavily unbalanced — only about 37% of rows (3.6M out of 9.8M) have gsc_data_available IS TRUE. This means for roughly 6.2 million rows, I have no real Search Console signal to work with, so my clustering will only reflect the subset of pages/days where GSC data actually exists — not the full picture of all content activity in March. Early rows in the month may also be more likely to lack data if there's a sync delay, which could bias which pages appear "available" versus not.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.